# Mova EXAONE LoRA 재학습 — Google Colab

프로덕션 채팅 모델(EXAONE-3.5-2.4B + LoRA, 노트북 lora-server)을 **증강 데이터셋으로 다시 학습**하고,
**Gemini(교사)·베이스 모델과 나란히 비교 평가**한 뒤, 서빙용 **GGUF(Q5_K_M)** 로 내보낸다.

| 단계 | 내용 | 예상 시간(L4 기준) |
|---|---|---|
| 1~3 | 설치 · 드라이브 · 데이터 분할 | 3분 |
| 4 | 베이스·Gemini 답 생성(평가셋) | 5분 |
| 5 | 학습(에폭마다 평가 손실) → 최적 에폭 학생 답 생성 | 15~25분 |
| 6 | Gemini 심판 블라인드 비교 + 하드 체크 리포트 | 5분 |
| 7 | 전체 데이터 재학습 → 병합 → GGUF 변환·양자화 → 드라이브 저장 | 20~30분 |

### 회차 관리 (2026-09-22 도입)
설정 셀의 `VERSION_TAG`만 바꾸면 데이터셋 이름과 산출물 폴더가 함께 따라간다.
- 데이터셋: `chat_teacher_dataset_<태그>.jsonl`
- 산출물: `내 드라이브/qwen-training/out/<태그>/`
이전에는 모든 회차가 `out/` 한 폴더에 섞여 어느 어댑터가 어느 회차인지 알기 어려웠다.

### 시작 전 준비
1. **런타임 → 런타임 유형 변경 → GPU: L4 또는 A100** (T4도 되지만 2~3배 느림)
2. 왼쪽 🔑 **보안 비밀**에 `GEMINI_API_KEY` 추가 → *노트북 액세스* 켜기
3. 바탕화면 `mova/FT/mova-colab-<회차>` 폴더의 **`chat_teacher_dataset_<회차>.jsonl`** 을
   구글 드라이브 `내 드라이브/qwen-training/` 에 올려 두기 (안 올려도 3번 셀이 업로드 창을 띄움)
4. **런타임 → 모두 실행**. 중간에 드라이브 권한 팝업만 승인하면 끝까지 돈다.
   드라이브 연결이 실패(`mount failed`)해도 멈추지 않는다 — 3번 셀이 업로드 창을 띄우고
   산출물을 `/content/qwen-training/out/<회차>/`에 쌓은 뒤, 마지막 8번 셀이 어댑터·리포트·GGUF를
   브라우저로 내려받게 한다(2026-09-27 추가). 이 경우 세션이 끊기면 산출물도 사라지니 8번 셀까지 꼭 실행.

### 2026-09-22 회차에서 달라진 것
- 데이터셋에 **멀티턴(대화 이어가기) 예시**가 들어갔다 — 기존 470행은 전부 단일턴이라
  "다른건", "고마워" 같은 후속 발화가 학습된 적이 없었다(서빙 프롬프트는 최근 6턴을 넣는데도).
- **평가셋에도 멀티턴을 포함**한다. 09-17에는 단일턴만 평가해 "인사에 3편 추천" 퇴행을 놓쳤다.
- 하드 체크에 **`nopick_ok`** 추가 — 정답이 "추천 안 함"인데 추천하면 실패. 기존 `empty_ok`는
  카탈로그가 비었을 때만 봐서 그 퇴행을 그냥 통과시켰다.
- 09-17 어댑터(`mova_20260917_124713`)는 **폐기**한다. q/k/v 3개 층만 학습된 회차다.

산출물은 `내 드라이브/qwen-training/out/<회차>/` 에 쌓인다. **프로덕션 반영은 이 노트북에서
하지 않는다** — GGUF를 내려받아 로컬에서 lora-server 교체·회귀 비교를 거친다: `mova_<시각>-Q5_K_M.gguf`, `mova_<시각>-adapter/`, `eval_report_<시각>.json`.
### v5 회차에서 달라진 것 (2026-09-23)
- 데이터 `chat_teacher_dataset_v5.jsonl` — v4에서 `v3_honest_genre`(카탈로그에 12편 있어도
  "장르에 맞는 게 없다"→0편) 48행 제거, `v3_honest`(배우 불일치) 30→12행. 연도 범위·장르/
  키워드 질의 긍정 예시를 실제 서빙 파이프라인+Gemini 교사로 보강. 이유: v4가 서빙에서
  `2000년대 초반 한국 영화`(후보 6편)·`법정 드라마`(후보 16편)를 0편으로 비움 — "확신 없으면
  0편" 과잉 일반화가 v3 배우 → v4 연도/장르로 옮겨 다닌 것.
- 하드 체크 **`ref_pick_ok`** 추가 — 정답이 1편 이상인데 학생이 0편이면 실패(`nopick_ok`의
  반대 방향). 이 수치가 v4 대비 올라야 합격.
- 합격 기준(운영): `eval_chat_queries.py` 23/23 목표(v4 21/23 — 연도·법정 2건 회복 + 배우 유지),
  `eval_chat_multiturn.py` 6/6 유지. GGUF는 어댑터만 내려도 노트북에서 `export_mova_gguf.py`로
  3분이면 만든다(v4 때 확인).


In [ ]:
# 1) GPU 확인 + 라이브러리
# transformers는 5.5.0 고정 — EXAONE 원격 코드가 4.57은 import 실패, 5.13은 create_causal_mask 인자 불일치(09-17 실측)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# 코랩 기본 torchao 0.10은 peft 0.20이 거부한다(0.16+ 요구) — 학습에 안 쓰므로 제거
!pip uninstall -y -q torchao
!pip install -q "transformers==5.5.0" "peft==0.20.0" accelerate google-genai sentencepiece

In [ ]:
# 2) 설정 — 보통은 건드릴 필요 없음
MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"   # 프로덕션과 같은 베이스(fp16 원본)
DRIVE_DIR = "/content/drive/MyDrive/qwen-training"   # 데이터셋을 올려둔 폴더
# 회차 태그 — 산출물이 out/<태그>/ 아래로 따로 쌓인다. 회차를 올릴 때 이 값만 바꾼다.
# (v2·v3 어댑터가 out/ 한 폴더에 섞여 어느 게 어느 회차인지 헷갈렸다, 2026-09-22)
VERSION_TAG = "v5"
DATA_NAME = f"chat_teacher_dataset_{VERSION_TAG}.jsonl"
ADAPTER_DIR = "/content/adapters"

HOLDOUT = 14          # 평가용 원본 질의 수 — 그 변형본까지 통째로 학습에서 뺀다(누수 방지)
EPOCHS = 3            # 최대 에폭. 평가 손실이 가장 낮은 에폭을 자동 선택
LR = 1e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05       # 09-09 프로덕션 어댑터와 동일
# EXAONE 선형층 이름(Llama식 o_proj/gate_proj 등이 아님) — 기존 설정은 q/k/v만 걸렸다
TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj", "c_fc_0", "c_fc_1", "c_proj"]
MAX_NEW_TOKENS = 320
GEMINI_MODEL = "gemini-3.1-flash-lite"               # 데이터셋을 만든 교사와 같은 모델
GEMINI_SLEEP = 4.5    # 무료 티어 분당 15회. 유료 키면 0.5로 줄여도 된다
FULL_RETRAIN = True   # 평가 후 평가셋까지 포함한 전체로 다시 학습해 내보낼지
QUANT = "Q5_K_M"      # 프로덕션 서빙 양자화와 동일
LLAMA_CPP_COMMIT = "304665fe7ac957df95e3ff8c8c4ffdf92dd6ffa3"  # 노트북 서빙 llama.cpp와 동일 커밋
SEED = 20260917

import torch
DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
EXPORT_DTYPE = torch.float16

In [ ]:
# 3) 드라이브 연결 + 데이터셋 + Gemini 키
import os, shutil
from google.colab import drive, files, userdata

# 드라이브 연결 — 권한 팝업을 안 끝내거나 팝업이 막히면 ValueError("mount failed")가 난다(2026-09-27 실측).
# 실패해도 학습은 진행한다: 로컬 /content에 쌓고 마지막 셀에서 내려받는다.
DRIVE_OK = False
for _try in range(2):
    try:
        drive.mount("/content/drive", force_remount=bool(_try))
        DRIVE_OK = True
        break
    except Exception as e:
        print(f"드라이브 연결 실패 {_try + 1}/2: {e}")
if not DRIVE_OK:
    DRIVE_DIR = "/content/qwen-training"
    print(f"⚠ 드라이브 없이 진행합니다 — 산출물은 {DRIVE_DIR}/out/{VERSION_TAG} 에 쌓이고 마지막 셀에서 내려받습니다.")
OUT_DIR = f"{DRIVE_DIR}/out/{VERSION_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)
DATA = f"{DRIVE_DIR}/{DATA_NAME}"
if not os.path.exists(DATA):
    print(f"{DATA} 가 없어 업로드 창을 엽니다 — 바탕화면 mova/FT/mova-colab-<회차> 폴더의 {DATA_NAME}을 고르세요")
    uploaded = files.upload()
    shutil.move(next(iter(uploaded)), DATA)
# 보안 비밀은 코랩 UI가 응답해야 넘어온다 — 토글이 꺼져 있거나 "모두 실행" 중
# 탭을 벗어나면 TimeoutException이 난다. 그때는 직접 입력으로 받는다.
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print(f"보안 비밀을 못 읽었습니다({type(e).__name__}) — 키를 직접 입력하세요")
    from getpass import getpass

    GEMINI_API_KEY = getpass("GEMINI_API_KEY: ").strip()
assert GEMINI_API_KEY, "GEMINI_API_KEY가 비어 있습니다"

In [ ]:
# 3-1) 평가셋 분리 — 원본(src) 단위로 떼어내 증강본이 평가에 새지 않게 한다
import json, random, re, time
from datetime import datetime, timezone

rows = [json.loads(line) for line in open(DATA, encoding="utf-8") if line.strip()]
orig = {r["src"]: r for r in rows if r["aug"] == "orig"}
rng = random.Random(SEED)
nopick = sorted(s for s, r in orig.items() if not json.loads(r["completion"])["picks"])
haspick = sorted(s for s in orig if s not in nopick)
eval_src = set(rng.sample(nopick, 2) + rng.sample(haspick, HOLDOUT - 2))  # 잡담·빈 카탈로그 2건 포함
train_rows = [r for r in rows if r["src"] not in eval_src]
# 평가셋은 원본 + 그 원본에서 파생된 멀티턴 행 — 09-17은 단일턴만 평가해
# "인사에 3편 추천" 퇴행을 놓쳤다. 후속 턴도 같이 재야 한다.
eval_rows = [orig[s] for s in sorted(eval_src)] + [
    r for r in rows if r["src"] in eval_src and r["aug"].startswith("mt_")
]
def rkey(r):
    return f'{r["src"]}:{r["aug"]}'   # 한 원본에서 여러 평가 행이 나오므로 src만으론 부족
n_mt = sum(1 for r in eval_rows if r["aug"].startswith("mt_"))
print(f"전체 {len(rows)}행(원본 {len(orig)}건) → 학습 {len(train_rows)}행 / "
      f"평가 {len(eval_rows)}행(원본 {len(eval_rows) - n_mt} + 멀티턴 {n_mt})")

In [ ]:
# 3-2) 하드 체크 — 프로덕션 파서가 문제 삼는 것들. 하나라도 어기면 사용자에게 깨진 답이 간다
# 카탈로그 줄 형식(2026-09-22 v4): "- movie_id=N 제목 (연도) [장르] [태그] — 줄거리: …"
# 제목 뒤 괄호 연도는 항상 있고, 대괄호는 0~2개, 줄거리 꼬리는 선택이다.
CAT_RE = re.compile(r"^- movie_id=(\d+) (.+?) \((?:\d{4}|연도 미상)\)(?: \[[^\]]*\])*(?: — 줄거리: .*)?$", re.M)
USER_RE = re.compile(r"<<<USER_INPUT>>>(.*?)<<<END_USER_INPUT>>>", re.S)
HANJA_RE = re.compile(r"[\u4e00-\u9fff]")
YEAR_TAIL = re.compile(r"\s*\(\d{4}\)\s*$")

def user_query(prompt):
    return USER_RE.findall(prompt)[-1]

def catalog(prompt):
    return {int(m[1]): m[2] for m in CAT_RE.finditer(prompt)}

def parse(raw):
    s, e = raw.find("{"), raw.rfind("}")
    try:
        return json.loads(raw[s : e + 1])
    except ValueError:
        return None

def _mid(p):
    try:
        return int(p.get("movie_id"))
    except (TypeError, ValueError):
        return None

def hard_check(raw, prompt, reference=None):
    cat, d = catalog(prompt), parse(raw)
    ok = isinstance(d, dict) and isinstance(d.get("picks"), list) and bool(str(d.get("intro", "")).strip())
    picks = d["picks"] if ok else []
    ok = ok and all(isinstance(p, dict) for p in picks)
    r = {"json": ok}
    r["grounded"] = ok and all(_mid(p) in cat for p in picks)           # 카탈로그 밖 작품 금지
    # 앱은 제목을 DB 값으로 덮어쓰므로 "(2025)" 같은 연도 꼬리는 무시 — 다른 영화 제목이면 id 오짝
    r["title"] = r["grounded"] and all(YEAR_TAIL.sub("", str(p.get("title", ""))).strip() == cat[_mid(p)] for p in picks)
    r["hook40"] = ok and all(len(str(p.get("hook", ""))) <= 40 for p in picks)
    r["max3"] = ok and len(picks) <= 3
    r["empty_ok"] = ok and (bool(cat) or not picks)                   # 빈 카탈로그면 추천 금지
    # 정답이 "추천 안 함"인 경우(인사·감사 등)에 추천하면 실패 — 카탈로그가 차 있어도
    # empty_ok는 통과하므로 09-17 인사 퇴행이 하드 체크를 그냥 지나갔다.
    ref = parse(reference) if reference else None
    r["nopick_ok"] = ok and not (ref is not None and not ref.get("picks") and picks)
    # v5(2026-09-23): 정답(교사)이 1편 이상 골랐는데 학생이 0편이면 실패 — v3 배우·v4 연도/장르
    # 회귀("후보가 있는데도 비움")를 홀드아웃에서 직접 잰다. nopick_ok의 반대 방향.
    r["ref_pick_ok"] = ok and not (ref is not None and ref.get("picks") and not picks)
    r["no_hanja"] = not HANJA_RE.search(raw)
    r["pass"] = all(r.values())
    return r

ref_pass = sum(hard_check(r["completion"], r["prompt"], r["completion"])["pass"] for r in rows) / len(rows)
print(f"정답(교사) 데이터 하드 체크 통과율: {ref_pass:.1%}  ← 체크 로직 자체 검증용")

In [ ]:
# 3-3) Gemini 호출(재시도 포함)
from google import genai

_client = genai.Client(api_key=GEMINI_API_KEY)

def gemini(prompt):
    for attempt in range(4):
        try:
            text = _client.models.generate_content(model=GEMINI_MODEL, contents=prompt).text or ""
            time.sleep(GEMINI_SLEEP)
            return text
        except Exception as e:  # 쿼터·일시 오류 — 점점 길게 쉬고 재시도
            print(f"  Gemini 재시도 {attempt + 1}/4: {e}")
            time.sleep(15 * (attempt + 1))
    return ""

In [ ]:
# 3-4) 모델·토크나이저 — 학습 프롬프트 형식은 프로덕션(train_mova_lora.py)과 동일한 chat template
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def load_base(dtype=None):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=dtype or DTYPE, device_map=DEVICE, trust_remote_code=True
    )
    # EXAONE 원격 코드에 없는 함수 보충 — peft·gradient checkpointing이 호출한다(임베딩 층 이름은 wte)
    type(model.transformer).get_input_embeddings = lambda self: self.wte
    return model

def chat_ids(prompt):
    return tok.apply_chat_template(
        [{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True
    )["input_ids"]

# 답 뒤에 붙는 턴 종료 꼬리("[|endofturn|]\n")를 템플릿에서 뽑아 둔다
_p = tok.apply_chat_template([{"role": "user", "content": "x"}], add_generation_prompt=True, tokenize=False)
_f = tok.apply_chat_template([{"role": "user", "content": "x"}, {"role": "assistant", "content": "Y"}], tokenize=False)
assert _f.startswith(_p + "Y"), "chat template 구조가 예상과 다름"
TURN_END = _f[len(_p) + 1 :]

@torch.no_grad()
def generate(model, prompt):
    # 서빙(llama-server temperature 0)과 맞춰 greedy
    model.eval()
    ids = chat_ids(prompt).to(model.device)
    out = model.generate(
        input_ids=ids, attention_mask=torch.ones_like(ids),
        max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
    )
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def free():
    # 호출 전에 del로 변수를 지워야 GPU 메모리가 실제로 풀린다
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. 비교 기준 만들기 — 학습 전 베이스 모델과 Gemini가 평가셋에 낸 답

In [ ]:
results = {rkey(r): {"query": user_query(r["prompt"]), "prompt": r["prompt"], "reference": r["completion"]}
           for r in eval_rows}
base = load_base()
for i, r in enumerate(eval_rows, 1):
    res = results[rkey(r)]
    res["base"] = generate(base, r["prompt"])
    res["gemini"] = gemini(r["prompt"])
    print(f"[{i}/{len(eval_rows)}] {res['query']}")
del base; free()

## 5. 학습 — 매 에폭 평가 손실을 재고, 가장 낮은 에폭의 어댑터로 학생 답을 만든다

In [ ]:
def example(r):
    # 프롬프트와 답을 따로 토큰화해 잇는다. 통째로 토큰화하면 "]"+"{"가 "]{" 한 토큰으로 합쳐져
    # 추론(프롬프트가 "]"로 끝나고 "{"부터 생성)과 다른 토큰열을 학습하게 된다(09-17 실측)
    p = chat_ids(r["prompt"])
    c = tok(r["completion"] + TURN_END, add_special_tokens=False, return_tensors="pt")["input_ids"]
    f = torch.cat([p, c], dim=1)
    labels = f.clone()
    labels[:, : p.shape[1]] = -100   # 프롬프트는 손실 제외, 답(JSON)만 학습
    return f, labels

def train(train_set, epochs, eval_set=None, tag="run"):
    model = get_peft_model(load_base(), LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGETS, task_type="CAUSAL_LM"))
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.print_trainable_parameters()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    data = [example(r) for r in train_set]
    eval_data = [example(r) for r in (eval_set or [])]
    order = random.Random(SEED)
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        order.shuffle(data)   # 같은 원본의 변형본이 연달아 나오지 않게
        total = 0.0
        t0 = time.time()
        for i, (ids, labels) in enumerate(data, 1):
            loss = model(input_ids=ids.to(model.device), labels=labels.to(model.device), use_cache=False).loss
            loss.backward()
            opt.step()
            opt.zero_grad()
            total += loss.item()
            if i % 50 == 0 or i == len(data):
                print(f"  epoch {ep} {i}/{len(data)} loss={total / i:.4f} ({time.time() - t0:.0f}s)")
        row = {"epoch": ep, "train_loss": round(total / len(data), 4)}
        if eval_data:
            model.eval()
            with torch.no_grad():
                row["eval_loss"] = round(sum(
                    model(input_ids=i_.to(model.device), labels=l_.to(model.device), use_cache=False).loss.item()
                    for i_, l_ in eval_data) / len(eval_data), 4)
        model.save_pretrained(f"{ADAPTER_DIR}/{tag}_ep{ep}")
        history.append(row)
        print(row)
    return model, history

In [ ]:
model, history = train(train_rows, EPOCHS, eval_rows, tag="eval")
del model; free()
BEST_EPOCH = min(history, key=lambda h: h["eval_loss"])["epoch"]
print(f"최적 에폭: {BEST_EPOCH}")

student = PeftModel.from_pretrained(load_base(), f"{ADAPTER_DIR}/eval_ep{BEST_EPOCH}")
for i, r in enumerate(eval_rows, 1):
    results[rkey(r)]["student"] = generate(student, r["prompt"])
    print(f"[{i}/{len(eval_rows)}] {results[rkey(r)]['query']}")
del student; free()

## 6. 비교 평가 — 하드 체크 + Gemini 블라인드 심판
- **하드 체크**: JSON · 카탈로그 밖 작품 · 제목 일치 · hook 40자 · 최대 3편 · 빈 카탈로그 추천 금지 · 한자
- **심판**: 같은 질의에 대한 두 답을 A/B 순서를 섞어 보여 주고 10점 만점 채점(루브릭은 RS 교사 루프와 동일 우선순위)

In [ ]:
JUDGE_PROMPT = '''당신은 영화 추천 챗봇 응답의 심사위원입니다. 같은 요청에 대한 두 응답 A와 B를 비교하세요.

채점 기준(중요한 순):
1. 그라운딩 — 카탈로그 안의 작품만, 질의 조건(장르·분위기·배우·연도)에 실제로 맞는 픽인가
2. intro 말투 — 존댓말, 1~2문장, 과장·확신 남발 금지, 자연스러운 한국어
3. hook — 각 픽의 한 줄 어필이 구체적이고 스포일러 없이 매력적인가(40자 이내)
4. 정직성 — 맞는 작품이 부족하면 억지로 채우지 않는가, 인사·잡담에는 추천하지 않는가
형식(JSON만 출력) 위반은 크게 감점하세요.

[사용자 질의]
{query}

[카탈로그]
{catalog}

[응답 A]
{a}

[응답 B]
{b}

JSON 한 줄만 출력:
{{"score_a": <0~10>, "score_b": <0~10>, "winner": "A" 또는 "B" 또는 "tie", "reason": "<한 문장>"}}'''

PAIRS = [("student", "gemini"), ("student", "base")]
flip = random.Random(SEED)

def judge(res, x, y):
    swap = flip.random() < 0.5
    a, b = (res[y], res[x]) if swap else (res[x], res[y])
    cat = "\n".join(f"- {k}: {v}" for k, v in catalog(res["prompt"]).items()) or "(비어 있음)"
    d = parse(gemini(JUDGE_PROMPT.format(query=res["query"], catalog=cat, a=a, b=b))) or {}
    try:
        sa, sb = float(d["score_a"]), float(d["score_b"])
    except (KeyError, TypeError, ValueError):
        return None
    w = {"A": "b" if swap else "a", "B": "a" if swap else "b"}.get(str(d.get("winner")), "tie")
    sx, sy = (sb, sa) if swap else (sa, sb)
    return {"x": sx, "y": sy, "winner": {"a": x, "b": y}.get(w, "tie"), "reason": d.get("reason", "")}

for i, res in enumerate(results.values(), 1):
    res["judge"] = {f"{x}_vs_{y}": judge(res, x, y) for x, y in PAIRS}
    print(f"[{i}/{len(results)}] {res['query']}")

In [ ]:
SYSTEMS = ["base", "student", "gemini", "reference"]
KEYS = ["pass", "json", "grounded", "title", "hook40", "max3", "empty_ok", "nopick_ok", "ref_pick_ok", "no_hanja"]
checks = {s: [hard_check(res[s], res["prompt"], res["reference"]) for res in results.values()] for s in SYSTEMS}
for res in results.values():
    res["checks"] = {s: hard_check(res[s], res["prompt"], res["reference"]) for s in SYSTEMS}

print("■ 하드 체크 통과율 (평가셋 %d건)" % len(results))
print(f"{'':10}" + "".join(f"{k:>10}" for k in KEYS))
summary = {"hard": {}, "judge": {}}
for s in SYSTEMS:
    rates = {k: sum(c[k] for c in checks[s]) / len(checks[s]) for k in KEYS}
    summary["hard"][s] = rates
    print(f"{s:10}" + "".join(f"{rates[k]:>10.0%}" for k in KEYS))

print("\n■ Gemini 심판 (블라인드, 10점 만점)")
for x, y in PAIRS:
    js = [res["judge"][f"{x}_vs_{y}"] for res in results.values()]
    js = [j for j in js if j]
    if not js:
        print(f"{x} vs {y}: 심판 응답 없음")
        continue
    agg = {
        f"{x}_avg": round(sum(j["x"] for j in js) / len(js), 2),
        f"{y}_avg": round(sum(j["y"] for j in js) / len(js), 2),
        f"{x}_win": sum(j["winner"] == x for j in js),
        "tie": sum(j["winner"] == "tie" for j in js),
        f"{y}_win": sum(j["winner"] == y for j in js),
        "judged": len(js),
    }
    summary["judge"][f"{x}_vs_{y}"] = agg
    print(f"{x} vs {y}: {agg}")

STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
report = {"stamp": STAMP, "best_epoch": BEST_EPOCH, "history": history, "summary": summary,
          "config": {"model": MODEL_ID, "epochs": EPOCHS, "lr": LR, "r": LORA_R, "alpha": LORA_ALPHA,
                     "train_rows": len(train_rows), "eval": len(eval_rows)},
          "results": {str(k): {kk: vv for kk, vv in v.items() if kk != "prompt"} for k, v in results.items()}}
with open(f"{OUT_DIR}/eval_report_{STAMP}.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1)
print(f"\n리포트 저장: {OUT_DIR}/eval_report_{STAMP}.json")

print("\n■ 샘플 3건")
for res in list(results.values())[:3]:
    print(f"\n질의: {res['query']}")
    for s in ("student", "gemini"):
        print(f"  [{s}] {res[s][:200]}")

### 내보낼지 판단 기준
- **학생 `pass` 가 베이스보다 높고 90% 이상**이어야 한다 — 그라운딩·형식이 깨지면 프로덕션에서 바로 티가 난다.
- **student vs gemini 평균 점수 차가 1.5점 이내**면 교사 품질에 근접한 것으로 본다.
- 기준 미달이면 7번 셀을 실행하지 말고 리포트 JSON의 `results` 에서 실패 사례를 먼저 본다.
- 이 평가는 학습 전후 비교가 아니라 교사와의 거리다. **이전 프로덕션 어댑터(09-09)와의 비교는
  노트북에 올린 뒤 `scripts/eval_chat_queries.py` 로 한다.**

## 7. 전체 데이터로 재학습 → 병합 → GGUF 변환·양자화 → 드라이브 저장

In [ ]:
import subprocess, sys

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

VERSION = "mova_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
if FULL_RETRAIN:
    model, _ = train(rows, BEST_EPOCH, tag="full")   # 평가셋 14건까지 포함, 검증된 에폭 수로
    del model; free()
    adapter = f"{ADAPTER_DIR}/full_ep{BEST_EPOCH}"
else:
    adapter = f"{ADAPTER_DIR}/eval_ep{BEST_EPOCH}"
shutil.copytree(adapter, f"{OUT_DIR}/{VERSION}-adapter", dirs_exist_ok=True)

MERGED = f"/content/{VERSION}-merged"
merged = PeftModel.from_pretrained(load_base(EXPORT_DTYPE), adapter).merge_and_unload()
merged.save_pretrained(MERGED, max_shard_size="1GB")
tok.save_pretrained(MERGED)
del merged; free()

LLAMA = "/content/llama.cpp"
if not os.path.exists(f"{LLAMA}/build/bin/llama-quantize"):
    sh(f"rm -rf {LLAMA} && git init -q {LLAMA}")
    sh(f"cd {LLAMA} && git fetch -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_COMMIT} && git checkout -q FETCH_HEAD")
    sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null")
    sh(f"cmake --build {LLAMA}/build --target llama-quantize -j{os.cpu_count()} > /dev/null")

F16 = f"/content/{VERSION}-f16.gguf"
OUT_GGUF = f"/content/{VERSION}-{QUANT}.gguf"

# convert_hf_to_gguf.py를 그대로 쓰면 안 된다 — llama.cpp 304665f의 ExaoneModel이
# `exaone.attention.layer_norm_rms_epsilon`을 안 써서 f16은 만들어지지만
# llama-quantize가 "key not found"로 죽는다(2026-09-17·09-22 코랩 재현).
# 실행 시점에 그 키만 한 줄 보충한다(노트북 scripts/convert_exaone_gguf.py와 동일).
_patch = f'''
import runpy, sys
llama = {LLAMA!r}
sys.path[:0] = [llama, llama + "/gguf-py"]
from conversion import exaone
_orig = exaone.ExaoneModel.set_gguf_parameters
def _p(self):
    _orig(self)
    self.gguf_writer.add_layer_norm_rms_eps(self.hparams["layer_norm_epsilon"])
exaone.ExaoneModel.set_gguf_parameters = _p
sys.argv = [llama + "/convert_hf_to_gguf.py", {MERGED!r}, "--outfile", {F16!r}, "--outtype", "f16"]
runpy.run_path(sys.argv[0], run_name="__main__")
'''
with open("/content/convert_exaone.py", "w") as f:
    f.write(_patch)
sh(f"{sys.executable} /content/convert_exaone.py")
sh(f"{LLAMA}/build/bin/llama-quantize {F16} {OUT_GGUF} {QUANT}")
shutil.copy(OUT_GGUF, OUT_DIR)
print(f"\n완료: {OUT_DIR}/{os.path.basename(OUT_GGUF)} ({os.path.getsize(OUT_GGUF) / 1e9:.2f} GB)")

## 8. 산출물 내려받기 — 드라이브 없이 돌렸을 때(연결됐으면 경로만 출력)

In [ ]:
# 드라이브가 연결됐으면 out/<회차>/에 이미 있으므로 경로만 알려 준다.
# 아니면 어댑터+리포트를 zip 하나로, GGUF는 따로 내려받는다(GGUF 1.7GB — 브라우저 다운로드가 끊기면
# 어댑터 zip만 받아도 된다: 노트북(teagy)에서 export_mova_gguf.py로 3분이면 변환된다).
import glob, zipfile

print(f"산출물 폴더: {OUT_DIR}")
for p in sorted(glob.glob(f"{OUT_DIR}/*")):
    size = f"{os.path.getsize(p) / 1e6:.1f} MB" if os.path.isfile(p) else "(폴더)"
    print(f"  {os.path.basename(p)}  {size}")

if DRIVE_OK:
    print(f"드라이브에 저장돼 있습니다 — 내 드라이브/qwen-training/out/{VERSION_TAG}/")
else:
    small = f"/content/{VERSION_TAG}-adapter-and-report.zip"
    with zipfile.ZipFile(small, "w", zipfile.ZIP_DEFLATED) as z:
        for p in glob.glob(f"{OUT_DIR}/**/*", recursive=True):
            if os.path.isfile(p) and not p.endswith(".gguf"):
                z.write(p, os.path.relpath(p, OUT_DIR))
    print(f"어댑터+리포트 zip 내려받기: {small} ({os.path.getsize(small) / 1e6:.1f} MB)")
    files.download(small)
    for g in glob.glob(f"{OUT_DIR}/*.gguf"):
        print(f"GGUF 내려받기: {g} ({os.path.getsize(g) / 1e9:.2f} GB) — 실패하면 어댑터 zip만으로 진행")
        files.download(g)
